In [1]:
from idlelib.rpc import response_queue

from dotenv import load_dotenv

load_dotenv()

True

## Creating Subagents

In [10]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculates the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculates the square of a number"""
    return x ** 2

@tool
def cube(x: float) -> float:
    """Calculates the cube of a number"""
    return x ** 3

In [11]:
from langchain.agents import create_agent

## create subagents

subagent_1 = create_agent(
    model="gpt-5-nano",
    tools=[square_root]
)

subagent_2 = create_agent(
    model="gpt-5-nano",
    tools=[square]
)

subagent_3 = create_agent(
    model="gpt-5-nano",
    tools=[cube]
)

## Calling Subagents

In [12]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages":[HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages":[HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_3(x: float) -> float:
    """Call subagent 3 in order to calculate the cube of a number"""
    response = subagent_3.invoke({"messages":[HumanMessage(content=f"Calculate the cube of {x}")]})
    return response["messages"][-1].content

# Creating the main agent

main_agent = create_agent(
    model="gpt-5-nano",
    tools=[call_subagent_1, call_subagent_2, call_subagent_3],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root, square and cube of a given number."
)

## Test

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages":[HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='7ea2fb9d-d8d3-4a32-9f2e-bcf1225a6010'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1310, 'prompt_tokens': 202, 'total_tokens': 1512, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1280, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7RmHR0hKSHLJKiRq9fp5TZKHYEel', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4401-072f-7be3-af83-8beae4bfd114-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_oEXzzEQCALjpgdibH49knOm5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202,

In [8]:
print(response["messages"][-1].content)

The square root of 456 is about 21.3541565041. (Since 456 = 4 × 114, sqrt(456) = 2√114.) If you want fewer decimals, it's approximately 21.3542.


In [13]:
question = "What is the cube of 99?"

response = main_agent.invoke({"messages":[HumanMessage(content=question)]})

In [14]:
pprint(response["messages"][-1].content)

'970299'
